## Point-output ablation (RQ1 support)

This notebook runs the point-output ablation requested for RQ1: the same TCN
encoder as MT-TrajNet, but with a plain linear output head trained with MSE
instead of the evidential head. Everything else (folds, seeds, normalisation,
stride-2, MAXLEN 6000) is identical to the authoritative Notebook 8 run, so the
result is directly comparable.

**Purpose:** to separate whether the hardness shortfall against XGBoost comes
from the evidential loss or from the shared trajectory encoder itself.

### Cell 1 — Environment, data load, and integrity check

Sets the seed for reproducibility, loads the assembled trajectories, and asserts
the MD5 fingerprint matches the authoritative data file
(`7d7fc76be1e4940198b76d9d0797a3a9`). If the match prints `True`, the input data
is the correct authoritative version and nothing stale has been loaded.

In [1]:
import numpy as np, pickle, torch, torch.nn as nn, torch.nn.functional as F, random, time, hashlib
from sklearn.metrics import mean_squared_error

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device,"| gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"
with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]; y_arr=data["y_arr"]; groups=data["groups"]
print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))

h=hashlib.md5()
with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    for c in iter(lambda:fh.read(1<<20),b""): h.update(c)
print("pkl md5:",h.hexdigest(),"| matches authoritative:",h.hexdigest()=="7d7fc76be1e4940198b76d9d0797a3a9")

device: cuda | gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25
pkl md5: 7d7fc76be1e4940198b76d9d0797a3a9 | matches authoritative: True


### Cell 2 — Constants, batch preparation, and fold construction

Defines the fixed pipeline settings (stride 2, MAXLEN 6000) and the four
prediction targets, the `prep_batch` function that downsamples, standardises and
pads each trajectory, and the stratified fold assignment. Code 15 is held out
entirely for the out-of-distribution test, leaving 941 batches across three
folds with the high-cluster codes balanced 22/6/6.

In [2]:
STRIDE=2; MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]; a=(a-mean)/std
        if len(a)<maxlen:
            a=np.vstack([a,np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")])
        else:
            a=a[:maxlen]
        out.append(a)
    return torch.tensor(np.array(out,dtype="float32")).transpose(1,2)

OOD_CODE=15
fold_codes=[[1,2,4,6,7,9,13,18,25],[10,11,17,19,22,24],[3,5,8,12,14,16,20,21,23]]
n_splits=len(fold_codes)
fold_test_idx=[np.where(np.isin(groups,fold_codes[f]))[0] for f in range(n_splits)]
print("folds:",[len(x) for x in fold_test_idx],"| CV total",sum(len(x) for x in fold_test_idx),"| code 15 held out")

folds: [314, 313, 314] | CV total 941 | code 15 held out


### Cell 3 — Model and point-output training

Defines the shared TCN encoder (four residual blocks, dilations 1/2/4/8, 128
channels) and `PointTrajNet`, which pairs that encoder with a two-layer linear
head. The model is trained with plain MSE on standardised targets, using the
same optimiser, learning rate, batch size, early stopping and per-fold seeds as
the evidential run. Each fold's out-of-fold RMSE is reported per target, followed
by the mean across folds.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU(); self.drop=nn.Dropout(dropout); self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]; out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]; out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout); self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout); self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x); return x.mean(dim=2)

class PointTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,n_targets))
    def forward(self,x): return self.head(self.encoder(x))

def train_fold_point(tr,va,te,max_epochs=150,lr=5e-4,batch_size=16,patience=10,seed=SEED):
    xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
    xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
    Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd); Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
    Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
    ymean=y_arr[tr].mean(axis=0);ystd=y_arr[tr].std(axis=0)+1e-6
    ytr=torch.tensor((y_arr[tr]-ymean)/ystd,dtype=torch.float32)
    yva=torch.tensor((y_arr[va]-ymean)/ystd,dtype=torch.float32)
    torch.manual_seed(seed); model=PointTrajNet().to(device)
    opt=torch.optim.Adam(model.parameters(),lr=lr); lossfn=nn.MSELoss()
    best=float("inf");best_state=None;wait=0
    for ep in range(max_epochs):
        model.train(); perm=torch.randperm(len(Xtr))
        for i in range(0,len(Xtr),batch_size):
            idx=perm[i:i+batch_size]; xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
            opt.zero_grad(); loss=lossfn(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=sum(lossfn(model(Xva[i:i+batch_size].to(device)),yva[i:i+batch_size].to(device)).item()
                   for i in range(0,len(Xva),batch_size))
        if vl<best-1e-4: best=vl;best_state={k:v.cpu().clone() for k,v in model.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=patience:break
    model.load_state_dict(best_state);model.eval()
    with torch.no_grad():
        gs=[model(Xte[i:i+batch_size].to(device)).cpu().numpy() for i in range(0,len(Xte),batch_size)]
    return np.vstack(gs)*ystd+ymean

t0=time.time(); point_fold_rmse=[]
for f in range(n_splits):
    te=fold_test_idx[f]
    trv=np.concatenate([fold_test_idx[j] for j in range(n_splits) if j!=f])
    g=groups[trv];uniq=np.unique(g)
    rng=np.random.RandomState(SEED+f)
    val_codes=rng.choice(uniq,size=max(1,len(uniq)//8),replace=False)
    va=trv[np.isin(g,val_codes)]; tr=trv[~np.isin(g,val_codes)]
    pred=train_fold_point(tr,va,te,seed=SEED+f)
    rmse=[np.sqrt(mean_squared_error(y_arr[te,k],pred[:,k])) for k in range(4)]
    point_fold_rmse.append(rmse)
    print(f"fold {f+1} point-output RMSE:",{targets[k]:round(rmse[k],3) for k in range(4)})
point_fold_rmse=np.array(point_fold_rmse)
print("\npoint-output mean RMSE:",{targets[k]:round(point_fold_rmse[:,k].mean(),3) for k in range(4)})
print("time:",round(time.time()-t0),"s")

fold 1 point-output RMSE: {'dissolution_av': np.float64(4.114), 'tbl_av_hardness': np.float64(8.444), 'tbl_rsd_weight': np.float64(0.793), 'fct_tensile': np.float64(0.456)}
fold 2 point-output RMSE: {'dissolution_av': np.float64(3.249), 'tbl_av_hardness': np.float64(5.423), 'tbl_rsd_weight': np.float64(0.455), 'fct_tensile': np.float64(0.322)}
fold 3 point-output RMSE: {'dissolution_av': np.float64(3.72), 'tbl_av_hardness': np.float64(6.227), 'tbl_rsd_weight': np.float64(0.525), 'fct_tensile': np.float64(0.422)}

point-output mean RMSE: {'dissolution_av': np.float64(3.694), 'tbl_av_hardness': np.float64(6.698), 'tbl_rsd_weight': np.float64(0.591), 'fct_tensile': np.float64(0.4)}
time: 339 s


In [4]:
import json
point_ablation={
 "run":"point-output ablation (RQ1 support), same TCN encoder, linear head, MSE loss, stratified folds, seed 42",
 "setup":"identical encoder/folds/normalisation as MT-TrajNet; evidential head replaced by 2-layer linear head; plain MSE; cross-fitting not applicable (point output, no intervals)",
 "point_output_rmse_mean":{targets[k]:round(float(point_fold_rmse[:,k].mean()),3) for k in range(4)},
 "point_output_rmse_folds":{targets[k]:[round(float(x),3) for x in point_fold_rmse[:,k]] for k in range(4)},
 "reference_evidential_rmse":{"dissolution_av":3.261,"tbl_av_hardness":7.698,"tbl_rsd_weight":0.585,"fct_tensile":0.345},
 "reference_tuned_xgb_rmse":{"dissolution_av":3.243,"tbl_av_hardness":4.761,"tbl_rsd_weight":0.611,"fct_tensile":0.270},
 "note":"On hardness the point-output model (6.698) beats the evidential model (7.698), so the evidential loss costs about 1.0 RMSE on hardness. But point-output still trails tuned XGBoost (4.761) by about 1.9, so most of the hardness gap is the trajectory encoder and the low high-cluster training count, not the evidential objective. On dissolution, weight RSD and tensile the point-output model is equal or slightly worse, so the evidential loss does not hurt overall accuracy elsewhere."
}
with open("/kaggle/working/point_output_ablation.json","w") as fh:
    json.dump(point_ablation,fh,indent=2)
print(json.dumps(point_ablation,indent=2))

{
  "run": "point-output ablation (RQ1 support), same TCN encoder, linear head, MSE loss, stratified folds, seed 42",
  "setup": "identical encoder/folds/normalisation as MT-TrajNet; evidential head replaced by 2-layer linear head; plain MSE; cross-fitting not applicable (point output, no intervals)",
  "point_output_rmse_mean": {
    "dissolution_av": 3.694,
    "tbl_av_hardness": 6.698,
    "tbl_rsd_weight": 0.591,
    "fct_tensile": 0.4
  },
  "point_output_rmse_folds": {
    "dissolution_av": [
      4.114,
      3.249,
      3.72
    ],
    "tbl_av_hardness": [
      8.444,
      5.423,
      6.227
    ],
    "tbl_rsd_weight": [
      0.793,
      0.455,
      0.525
    ],
    "fct_tensile": [
      0.456,
      0.322,
      0.422
    ]
  },
  "reference_evidential_rmse": {
    "dissolution_av": 3.261,
    "tbl_av_hardness": 7.698,
    "tbl_rsd_weight": 0.585,
    "fct_tensile": 0.345
  },
  "reference_tuned_xgb_rmse": {
    "dissolution_av": 3.243,
    "tbl_av_hardness": 4.761,
 